# MarketMind — Fine-tuning on Google Colab

**Runtime required:** GPU — T4 minimum, A100 recommended (Colab Pro).

Fine-tunes `Qwen/Qwen3-8B` on the FinGPT Dow30 dataset using QLoRA.  
Edit **Section 2 (Config)** then run all cells top to bottom.

Checkpoints save to **Google Drive** so they survive session timeouts.

## 0. Check GPU

In [ ]:
!nvidia-smi

## 1. Install Dependencies

In [ ]:
%%capture
!pip install -q \
    transformers>=4.45.0 \
    datasets>=2.20.0 \
    peft>=0.12.0 \
    trl>=0.11.0 \
    bitsandbytes>=0.43.0 \
    accelerate>=0.34.0 \
    wandb \
    scikit-learn \
    huggingface_hub

## 2. Config — Edit This Cell to Switch Ablations

| Ablation | `LOAD_IN_4BIT` | `LOAD_IN_8BIT` | `LORA_R` | ~VRAM |
|---|---|---|---|---|
| qlora_4bit_r16 | True | False | 16 | ~18 GB |
| **qlora_4bit_r32** ← default | **True** | **False** | **32** | **~20 GB** |
| qlora_4bit_r64 | True | False | 64 | ~24 GB |
| lora_8bit_r16 | False | True | 16 | ~28 GB |
| lora_8bit_r32 | False | True | 32 | ~32 GB |
| lora_8bit_r64 | False | True | 64 | ~38 GB |
| full_finetune | False | False | — | ~65 GB A100 only |
| cls_head_r32 | True | False | 32 | ~20 GB |

> **T4 users:** set `LORA_R=16` and `BATCH_SIZE=1`

In [ ]:
# ── Edit this cell to switch ablations ───────────────────────────────────────

RUN_NAME       = "qlora_4bit_r32"   # used for output dir + WandB run name
MODEL_NAME     = "Qwen/Qwen3-8B"

# Quantization — set one True, or both False for full fine-tune
LOAD_IN_4BIT   = True
LOAD_IN_8BIT   = False

# LoRA
LORA_R         = 32
LORA_ALPHA     = 64     # convention: 2x rank
LORA_DROPOUT   = 0.05

# Training
NUM_EPOCHS     = 3
BATCH_SIZE     = 2      # reduce to 1 on T4
GRAD_ACCUM     = 8      # effective batch = BATCH_SIZE * GRAD_ACCUM
LEARNING_RATE  = 2e-4
MAX_SEQ_LEN    = 2048

# Classification head ablation (cls_head_r32 run only)
USE_CLS_HEAD   = False

# Smoke test: only 5 steps (~3 min) to verify everything works before a full run
SMOKE_TEST     = False

# WandB — set to None to disable
WANDB_PROJECT  = "marketmind"

# Google Drive output path
OUTPUT_DIR     = f"/content/drive/MyDrive/marketmind/runs/{RUN_NAME}"

# ─────────────────────────────────────────────────────────────────────────────
print(f"Run:        {RUN_NAME}")
print(f"4-bit:      {LOAD_IN_4BIT}  |  8-bit: {LOAD_IN_8BIT}")
print(f"LoRA rank:  {LORA_R}  |  alpha: {LORA_ALPHA}")
print(f"Epochs:     {NUM_EPOCHS}  |  batch: {BATCH_SIZE} (eff. {BATCH_SIZE * GRAD_ACCUM})")
print(f"Smoke test: {SMOKE_TEST}")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Checkpoints will save to: {OUTPUT_DIR}")

## 4. HuggingFace + WandB Login

In [ ]:
from huggingface_hub import login
login()  # paste your HF token

In [ ]:
import os

if WANDB_PROJECT:
    import wandb
    wandb.login()  # paste your WandB API key
    os.environ["WANDB_PROJECT"] = WANDB_PROJECT
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("WandB disabled.")

## 5. Load Dataset

In [ ]:
from datasets import load_dataset
from collections import Counter

ds = load_dataset("FinGPT/fingpt-forecaster-dow30-202305-202312", split="train")
dataset = ds.train_test_split(test_size=0.1, seed=42)

print(dataset)
print("\nLabel distribution:", Counter(dataset["train"]["label"]))

## 6. Prompt Formatting

The FinGPT dataset uses Llama-2 `[INST]` tags. We strip those and reformat as Qwen3 ChatML.

In [ ]:
import re

SYSTEM_PROMPT = (
    "You are a seasoned stock market analyst. Your task is to list the positive "
    "developments and potential concerns for companies based on relevant news and "
    "basic financial data from the past weeks, then make a prediction about the "
    "companies' stock price movement for the upcoming week.\n\n"
    "[Positive Developments]:\n1. ...\n\n"
    "[Potential Concerns]:\n1. ...\n\n"
    "[Prediction & Analysis]:\n..."
)

def strip_llama_tags(text):
    text = re.sub(r"\[/?INST\]", "", text)
    text = re.sub(r"<<SYS>>.*?<</SYS>>", "", text, flags=re.DOTALL)
    return text.strip()

def format_example(example):
    user = strip_llama_tags(example["prompt"])
    answer = example["answer"].strip()
    if USE_CLS_HEAD:
        answer = f"Prediction: {example['label']}\n\n" + answer
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )

# Sanity check — print one formatted example
print(format_example(dataset["train"][0])[:600])

## 7. Load Model + Tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, prepare_model_for_kbit_training

# Quantization config
bnb_config = None
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )
elif LOAD_IN_8BIT:
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)

# Load model
print(f"Loading {MODEL_NAME} ...")
model_kwargs = dict(
    pretrained_model_name_or_path=MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
if bnb_config:
    model_kwargs["quantization_config"] = bnb_config

model = AutoModelForCausalLM.from_pretrained(**model_kwargs)
if bnb_config:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded — {model.num_parameters():,} parameters")

In [ ]:
# LoRA config (skipped for full fine-tune)
peft_config = None
if LOAD_IN_4BIT or LOAD_IN_8BIT:
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )
    print(f"LoRA: rank={LORA_R}, alpha={LORA_ALPHA}")
else:
    print("Full fine-tune mode (no LoRA).")

## 8. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim="paged_adamw_32bit" if (LOAD_IN_4BIT or LOAD_IN_8BIT) else "adamw_torch",
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="wandb" if WANDB_PROJECT else "none",
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQ_LEN,
)

if SMOKE_TEST:
    sft_kwargs.update(max_steps=5, logging_steps=1, eval_steps=5, save_steps=5)
    print("[SMOKE TEST] Running 5 steps only.")

trainer_kwargs = dict(
    model=model,
    args=SFTConfig(**sft_kwargs),
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    formatting_func=format_example,
)
if peft_config:
    trainer_kwargs["peft_config"] = peft_config

trainer = SFTTrainer(**trainer_kwargs)
trainer.train()

In [ ]:
# Save to Drive
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to: {OUTPUT_DIR}")

## 9. Evaluate

Greedy decoding on 200 test examples → accuracy + macro F1.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.notebook import tqdm

_KEYWORDS = {
    "up": "up", "increase": "up", "rise": "up", "rising": "up",
    "higher": "up", "bullish": "up", "positive": "up", "growth": "up",
    "down": "down", "decrease": "down", "fall": "down", "falling": "down",
    "lower": "down", "bearish": "down", "negative": "down", "decline": "down",
    "hold": "neutral", "flat": "neutral", "neutral": "neutral", "stable": "neutral",
}

def extract_direction(text):
    match = re.search(r"\[Prediction\s*&\s*Analysis\].*?(\w+)", text, re.IGNORECASE | re.DOTALL)
    if match and match.group(1).lower() in _KEYWORDS:
        return _KEYWORDS[match.group(1).lower()]
    for kw, d in _KEYWORDS.items():
        if re.search(r"\b" + kw + r"\b", text[-300:].lower()):
            return d
    return "unknown"

def coarse_label(fingpt_label):
    l = fingpt_label.strip().lower()
    if l.startswith("up"):   return "up"
    if l.startswith("down"): return "down"
    return "neutral"

print("Helpers ready.")

In [ ]:
# Merge adapter for clean inference
if peft_config is not None:
    from peft import PeftModel
    print("Merging adapter into base model...")
    eval_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
    )
    eval_model = PeftModel.from_pretrained(eval_model, OUTPUT_DIR).merge_and_unload()
else:
    eval_model = model

eval_model.eval()
tokenizer.padding_side = "left"
print("Ready.")

In [ ]:
MAX_EVAL = 200
preds, refs = [], []
n = min(MAX_EVAL, len(dataset["test"]))

for i in tqdm(range(n)):
    ex = dataset["test"][i]
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{strip_llama_tags(ex['prompt'])}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    preds.append(extract_direction(generated))
    refs.append(coarse_label(ex["label"]))

LABELS = ["up", "down", "neutral"]
acc = accuracy_score(refs, preds)
f1  = f1_score(refs, preds, labels=LABELS, average="macro", zero_division=0)
print(f"\nRun: {RUN_NAME}  |  n={n}")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 Macro:  {f1:.4f}")
print(classification_report(refs, preds, labels=LABELS, zero_division=0))

## 10. Baseline Comparison (optional)

Uncomment to evaluate zero-shot Qwen3-8B for direct comparison against the fine-tuned model.

In [ ]:
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
# )
# base_model.eval()
# base_preds, base_refs = [], []
# for i in tqdm(range(n)):
#     ex = dataset["test"][i]
#     prompt = (
#         f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
#         f"<|im_start|>user\n{strip_llama_tags(ex['prompt'])}<|im_end|>\n"
#         f"<|im_start|>assistant\n"
#     )
#     inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(base_model.device)
#     with torch.no_grad():
#         out = base_model.generate(**inputs, max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.pad_token_id)
#     generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
#     base_preds.append(extract_direction(generated))
#     base_refs.append(coarse_label(ex["label"]))
# base_acc = accuracy_score(base_refs, base_preds)
# base_f1  = f1_score(base_refs, base_preds, labels=LABELS, average="macro", zero_division=0)
# print(f"Baseline  — Accuracy: {base_acc:.4f}  F1: {base_f1:.4f}")
# print(f"Finetuned — Accuracy: {acc:.4f}  F1: {f1:.4f}")
# print(f"Delta     — Accuracy: {acc-base_acc:+.4f}  F1: {f1-base_f1:+.4f}")

## 11. Save Results

In [ ]:
import json

results = {"run_name": RUN_NAME, "accuracy": acc, "f1_macro": f1, "n_samples": n}
path = f"{OUTPUT_DIR}/results.json"
with open(path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved: {path}")
print(json.dumps(results, indent=2))